### Trial: BiLSTM + Neutrosophic Loss with Hyperparameter Tuning

##### 1. Imports

In [ ]:
%pip install optuna

In [2]:
import kagglehub
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.utils import class_weight
from sklearn.metrics import f1_score
from collections import Counter
import optuna
import optuna.visualization as vis
from torch.amp import autocast, GradScaler
from tqdm import tqdm

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score

##### 2. Configuration

In [3]:
class Config:
    MAX_WORDS = 20000
    MAX_LEN = 128
    EMBEDDING_DIM = 128
    HIDDEN_DIM = 64
    BATCH_SIZE = 64
    LR = 1e-3
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    path = kagglehub.dataset_download("sandhyacvijay/beads-dataset")
    TRAIN_PATH = path + '/train_set.parquet'
    VAL_PATH = path + '/val_set.parquet'
    TEST_PATH = path + '/test_set.parquet'

Using Colab cache for faster access to the 'beads-dataset' dataset.


##### 3. Tokeniser

In [4]:
class SimpleTokenizer:
    def __init__(self, num_words):
        self.num_words = num_words
        self.word_index = {"<PAD>": 0, "<OOV>": 1}

    def fit_on_texts(self, texts):
        words = [word for text in texts for word in str(text).lower().split()]
        most_common = Counter(words).most_common(self.num_words - 2)
        for i, (word, _) in enumerate(most_common):
            self.word_index[word] = i + 2

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            seq = [self.word_index.get(w, 1) for w in str(text).lower().split()]
            sequences.append(seq)
        return sequences

##### 4. Dataset Class

In [5]:
class MediaBiasDataset(Dataset):
    def __init__(self, sequences, labels_df):
        self.sequences = sequences
        self.bias = labels_df['bias'].values
        self.sentiment = labels_df['sentiment'].values
        self.toxic = labels_df['toxic'].values

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        # Manual padding to MAX_LEN
        seq = self.sequences[idx][:Config.MAX_LEN]
        padded_seq = seq + [0] * (Config.MAX_LEN - len(seq))

        return {
            'text': torch.tensor(padded_seq, dtype=torch.long),
            'bias': torch.tensor(self.bias[idx], dtype=torch.long),
            'sentiment': torch.tensor(self.sentiment[idx], dtype=torch.long),
            'toxic': torch.tensor(self.toxic[idx], dtype=torch.long)
        }

##### 5. Neutrosophic Loss Function

In [6]:
class NeutrosophicLoss(nn.Module):
    def __init__(self, weight=None, alpha=1.0, beta=0.1, gamma=0.1):
        super(NeutrosophicLoss, self).__init__()
        self.weight = weight
        self.alpha = alpha  # Truth weight
        self.beta = beta    # Indeterminacy weight
        self.gamma = gamma  # Falsity weight

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)

        # Truth (T): Probability of the correct class
        T = probs.gather(1, targets.view(-1, 1)).squeeze()

        # Indeterminacy (I): Degree of uncertainty (1 - max probability)
        I = 1 - probs.max(dim=1)[0]

        # Falsity (F): Degree of assigned incorrect probability
        F_val = 1 - T

        # Base Cross-Entropy
        ce_loss = F.cross_entropy(logits, targets, weight=self.weight)

        # Final Neutrosophic Loss Calculation
        total_loss = (self.alpha * ce_loss) + (self.beta * I.mean()) + (self.gamma * F_val.mean())
        return total_loss

##### 6. Model Architecture

In [7]:
class SpatialDropout(nn.Module):
    def __init__(self, drop_prob):
        super().__init__()
        self.drop = nn.Dropout1d(drop_prob)

    def forward(self, x):
        # Input x shape: [Batch, Seq_Len, Embedding_Dim]
        # Change to [Batch, Embedding_Dim, Seq_Len]
        x = x.permute(0, 2, 1)
        x = self.drop(x)
        # Change back to [Batch, Seq_Len, Embedding_Dim]
        return x.permute(0, 2, 1)

class MultitaskBiLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, Config.EMBEDDING_DIM, padding_idx=0)
        self.spatial_dropout = SpatialDropout(0.2)
        self.lstm = nn.LSTM(Config.EMBEDDING_DIM, Config.HIDDEN_DIM,
                            bidirectional=True, batch_first=True)

        # Shared Dense Layer
        self.fc_shared = nn.Linear(Config.HIDDEN_DIM * 2, 64)
        self.relu = nn.ReLU()

        # Task Heads
        self.bias_head = nn.Linear(64, 3)
        self.sentiment_head = nn.Linear(64, 3)
        self.toxic_head = nn.Linear(64, 2)

    def forward(self, x):
        x = self.spatial_dropout(self.embedding(x))
        _, (h_n, _) = self.lstm(x)
        # Concatenate final forward and backward hidden states
        x = torch.cat((h_n[-2,:,:], h_n[-1,:,:]), dim=1)

        shared = self.relu(self.fc_shared(x))

        return {
            'bias': self.bias_head(shared),
            'sentiment': self.sentiment_head(shared),
            'toxic': self.toxic_head(shared)
        }

##### 7. Optuna Objective Function

In [8]:
def train_epoch_amp(model, loader, optimizer, loss_fns, scaler):
    model.train()
    epoch_loss = 0
    for batch in tqdm(loader, desc="Training", leave=False):
        optimizer.zero_grad()
        text = batch['text'].to(Config.DEVICE)

        # 1. Runs the forward pass with autocasting
        with autocast(device_type='cuda', dtype=torch.float16):
            out = model(text)
            l_bias = loss_fns['bias'](out['bias'], batch['bias'].to(Config.DEVICE))
            l_sent = loss_fns['sentiment'](out['sentiment'], batch['sentiment'].to(Config.DEVICE))
            l_toxic = loss_fns['toxic'](out['toxic'], batch['toxic'].to(Config.DEVICE))
            loss = l_bias + l_sent + l_toxic

        # 2. Scales the loss and calls backward()
        scaler.scale(loss).backward()

        # 3. scaler.step() first unscales the gradients.
        # If they are finite, it calls optimizer.step(), otherwise skips it.
        scaler.step(optimizer)

        # 4. Updates the scale for the next iteration
        scaler.update()

        epoch_loss += loss.item()
    return epoch_loss / len(loader)

In [10]:
def objective(trial):
    # 1. Suggesting alpha, beta, gamma
    alpha = trial.suggest_float("alpha", 0.01, 1.0)
    beta = trial.suggest_float("beta", 0.01, 1.0)
    gamma = trial.suggest_float("gamma", 0.01, 1.0)

    model = MultitaskBiLSTM(Config.MAX_WORDS).to(Config.DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=Config.LR)

    loss_fns = {task: NeutrosophicLoss(weight=weights[task], alpha=alpha, beta=beta, gamma=gamma)
                for task in ['bias', 'sentiment', 'toxic']}

    scaler = GradScaler()

    for epoch in range(3):
        train_loss = train_epoch_amp(model, train_loader, optimizer, loss_fns, scaler)

        # 2. Validation for Pruning
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                out = model(batch['text'].to(Config.DEVICE))
                # Optimizing based on Bias task Macro-F1
                val_preds.extend(torch.argmax(out['bias'], dim=1).cpu().numpy())
                val_targets.extend(batch['bias'].numpy())

        score = f1_score(val_targets, val_preds, average='macro')

        # 3. Optuna reporting
        trial.report(score, epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return score

##### 8. Main Execution

In [11]:
if __name__ == "__main__":
    train_df = pd.read_parquet(Config.TRAIN_PATH)
    val_df = pd.read_parquet(Config.VAL_PATH)

    tokenizer = SimpleTokenizer(Config.MAX_WORDS)
    tokenizer.fit_on_texts(train_df['text'])

    weights = {t: torch.tensor(class_weight.compute_class_weight('balanced', classes=np.unique(train_df[t]), y=train_df[t]),
                dtype=torch.float).to(Config.DEVICE) for t in ['bias', 'sentiment', 'toxic']}

    train_loader = DataLoader(MediaBiasDataset(tokenizer.texts_to_sequences(train_df['text']), train_df), batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(MediaBiasDataset(tokenizer.texts_to_sequences(val_df['text']), val_df), batch_size=Config.BATCH_SIZE)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=15)

    print("Best parameters:", study.best_params)
    print("Best Macro-F1:", study.best_value)

[I 2025-12-23 14:45:03,332] A new study created in memory with name: no-name-596d128b-3d3d-406d-9664-5782b2fff2cb
[I 2025-12-23 15:02:37,338] Trial 0 finished with value: 0.8958744756868618 and parameters: {'alpha': 0.5691976594048015, 'beta': 0.08913183277333704, 'gamma': 0.6166240128271506}. Best is trial 0 with value: 0.8958744756868618.
[I 2025-12-23 15:20:00,070] Trial 1 finished with value: 0.8924965939379247 and parameters: {'alpha': 0.8499333374934362, 'beta': 0.034837415025069596, 'gamma': 0.6302537162036194}. Best is trial 0 with value: 0.8958744756868618.
[I 2025-12-23 15:37:31,442] Trial 2 finished with value: 0.8936963775193462 and parameters: {'alpha': 0.7719842255682133, 'beta': 0.6029022587574228, 'gamma': 0.7576766051670275}. Best is trial 0 with value: 0.8958744756868618.
[I 2025-12-23 15:54:44,558] Trial 3 finished with value: 0.8913415469166729 and parameters: {'alpha': 0.2553379771015271, 'beta': 0.4769285656533473, 'gamma': 0.2519250964326153}. Best is trial 0 wit

Best parameters: {'alpha': 0.9743241045702373, 'beta': 0.01102759664672523, 'gamma': 0.633466849269729}
Best Macro-F1: 0.8993639245681099


##### 9. Evaluation

In [12]:
# 1. Parallel Coordinate Plot
fig1 = vis.plot_parallel_coordinate(study)
fig1.show()

# 2. Hyperparameter Importance Plot
fig2 = vis.plot_param_importances(study)
fig2.show()

# 3. Optimization History
fig3 = vis.plot_optimization_history(study)
fig3.show()

best_params = study.best_params
print(f"Optimal Alpha (Truth): {best_params['alpha']:.4f}")
print(f"Optimal Beta (Indeterminacy): {best_params['beta']:.4f}")
print(f"Optimal Gamma (Falsity): {best_params['gamma']:.4f}")

Optimal Alpha (Truth): 0.9743
Optimal Beta (Indeterminacy): 0.0110
Optimal Gamma (Falsity): 0.6335


##### 10. Save Results

In [13]:
with open("best_neutrosophic_params.txt", "w") as f:
    f.write(str(study.best_params))
    f.write(f"\nBest Macro-F1: {study.best_value}")

# Export the entire study history to a CSV
df_results = study.trials_dataframe()
df_results.to_csv("optuna_neutrosophic_study.csv", index=False)

# Save the study object itself
import joblib
joblib.dump(study, "neutrosophic_study.pkl")

['neutrosophic_study.pkl']